In [1]:
#Importamos librerias 
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 50)

## 1. Carga de datos y optimización de tipos

**Objetivo.** Construir un único DataFrame con los vuelos de enero a marzo de 2025 del BTS (*Reporting Carrier On-Time Performance*), conservando únicamente las variables necesarias para el análisis.

**Metodología.**
- Se cargan 35 de las 110 columnas originales (`usecols`), lo que reduce el tiempo de lectura y el consumo de memoria.
- `FlightDate` se interpreta como fecha en la propia lectura (`parse_dates`).
- Las variables de baja cardinalidad (aerolínea, aeropuertos, estados, ciudades, franja horaria, código de cancelación y matrícula) se leen como `category`, un tipo mucho más eficiente en memoria que el texto.
- Al concatenar los tres meses, pandas devuelve a `object` las columnas categóricas cuyo conjunto de categorías difiere entre meses (por ejemplo, una matrícula que solo aparece en febrero). Por ello se reconvierten a `category` sobre el DataFrame ya unido.
- Se liberan los DataFrames mensuales (`del meses_csv`) para evitar duplicar datos en memoria.

**Resultado.** DataFrame de 1.645.503 filas y 35 columnas.

In [2]:
#seleccionamos las columnas que queremos para nuestros analisis 
columnas = ["Year", "Month", "DayofMonth", "DayOfWeek", "FlightDate", "Reporting_Airline","Tail_Number", "Flight_Number_Reporting_Airline", "Origin", "OriginCityName","OriginState", "Dest", "DestCityName", "DestState", "CRSDepTime", "DepTime", "DepDelay","DepDel15", "DepTimeBlk", "TaxiOut", "TaxiIn", "CRSArrTime", "ArrTime", "ArrDelay","ArrDel15", "Cancelled", "CancellationCode", "Diverted", "AirTime", "Distance", "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
cols_categoria = ["Reporting_Airline", "Origin", "Dest", "OriginState", "DestState",
                  "OriginCityName", "DestCityName", "DepTimeBlk", "CancellationCode", "Tail_Number"]

archivos = sorted(Path("../data/raw/bts").glob("*.csv"))

meses_csv = []
for archivo in archivos:
    df_mes = pd.read_csv(archivo,
                         usecols=columnas,
                         dtype={col: "category" for col in cols_categoria},
                         parse_dates=["FlightDate"])
    meses_csv.append(df_mes)

df = pd.concat(meses_csv, ignore_index=True)
del meses_csv
# El concat devuelve a object las categorías que difieren entre meses: las reconvertimos
cols_objeto = df.select_dtypes(include="object").columns
df[cols_objeto] = df[cols_objeto].astype("category")
print(df.shape)
df.info(memory_usage="deep")

(1645503, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1645503 entries, 0 to 1645502
Data columns (total 35 columns):
 #   Column                           Non-Null Count    Dtype         
---  ------                           --------------    -----         
 0   Year                             1645503 non-null  int64         
 1   Month                            1645503 non-null  int64         
 2   DayofMonth                       1645503 non-null  int64         
 3   DayOfWeek                        1645503 non-null  int64         
 4   FlightDate                       1645503 non-null  datetime64[ns]
 5   Reporting_Airline                1645503 non-null  category      
 6   Tail_Number                      1641401 non-null  category      
 7   Flight_Number_Reporting_Airline  1645503 non-null  int64         
 8   Origin                           1645503 non-null  category      
 9   OriginCityName                   1645503 non-null  category      
 10  OriginState     

### 2. Estado del vuelo y origen de los valores nulos

**Objetivo.** Determinar si los valores nulos de `TaxiOut`, `TaxiIn`, `AirTime` y `ArrTime` corresponden a errores de registro o a información estructural del proceso.

**Metodología.** Se crea la variable `estado` (`cancelado`, `desviado` o `normal`) a partir de `Cancelled` y `Diverted`, y se cuantifican los nulos de cada columna dentro de cada categoría.

**Criterio.** Un nulo en un vuelo que no se realizó es esperable, ya que no genera tiempos de rodaje ni de vuelo. Un nulo en un vuelo `normal` indicaría un dato ausente y requeriría investigación.

In [3]:
df["estado"] = np.select(
    [df["Cancelled"] == 1, df["Diverted"] == 1],
    ["cancelado", "desviado"],
    default="normal"
)

print(df["estado"].value_counts())

cols_revisar = ["TaxiOut", "TaxiIn", "AirTime", "ArrTime"]
df[cols_revisar].isna().groupby(df["estado"]).sum()

estado
normal       1611046
cancelado      30640
desviado        3817
Name: count, dtype: int64


,TaxiOut,TaxiIn,AirTime,ArrTime
estado,,,,
cancelado,30401,30640,30640,30640
desviado,0,612,3817,612
normal,0,0,0,0


### Conclusión: los nulos son estructurales

| Estado | Nº de vuelos | Nulos observados | Interpretación |
|---|---|---|---|
| Normal | 1.611.046 | Ninguno en las cuatro columnas | Registro completo. |
| Cancelado | 30.640 | `TaxiIn`, `AirTime` y `ArrTime` en todos; `TaxiOut` en 30.401 | El vuelo no llegó a realizarse. Los 239 con `TaxiOut` informado iniciaron el rodaje y se cancelaron después. |
| Desviado | 3.817 | `AirTime` en todos; `TaxiIn` y `ArrTime` en 612 | En 3.205 casos consta el aterrizaje, pero en un aeropuerto alternativo y no en `Dest`. |

**Decisiones metodológicas.**
1. No se imputan valores en estas cuatro columnas, dado que el nulo indica que el evento no ocurrió.
2. El análisis del retraso de llegada se restringe a los vuelos `normal`.
3. Los vuelos cancelados y desviados se analizan por separado, mediante sus tasas de cancelación y de desvío.
4. En los vuelos desviados, `ArrTime` y `TaxiIn` no son comparables con `Dest`, porque corresponden a otro aeropuerto.

## 3. Coherencia entre `DepTime` y `DepDelay`

**Objetivo.** Analizar los 111 vuelos que tienen hora real de salida (`DepTime`) pero carecen de retraso de salida (`DepDelay`), ya que ambas variables deberían informarse conjuntamente.

**Metodología.** Se aíslan mediante una máscara booleana, se tabula su `estado` y se inspecciona una muestra de 10 registros comparando la hora programada (`CRSDepTime`) con la real (`DepTime`).

In [4]:
# Se recrea 'estado' por si el DataFrame se ha vuelto a cargar
df["estado"] = np.select(
    [df["Cancelled"] == 1, df["Diverted"] == 1],
    ["cancelado", "desviado"],
    default="normal"
)

# 1) Vuelos con hora de salida pero sin retraso de salida
mask = df["DepTime"].notna() & df["DepDelay"].isna()
print("Vuelos sospechosos:", mask.sum())
print(df.loc[mask, "estado"].value_counts())

cols_ver = ["FlightDate", "Reporting_Airline", "Origin", "Dest", "CRSDepTime",
            "DepTime", "DepDelay", "TaxiOut", "estado", "CancellationCode"]
display(df.loc[mask, cols_ver].head(10))


Vuelos sospechosos: 111
estado
cancelado    111
Name: count, dtype: int64


,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,DepDelay,TaxiOut,estado,CancellationCode
151105,2025-01-02,OO,SFO,SAN,1654,2021.0,NaN,NaN,cancelado,B
157825,2025-01-16,OO,DTW,IAD,2140,2210.0,NaN,NaN,cancelado,B
168229,2025-01-04,OO,DEN,LNK,1730,2103.0,NaN,NaN,cancelado,B
168321,2025-01-04,OO,DEN,HOB,1738,1742.0,NaN,NaN,cancelado,B
169152,2025-01-05,OO,EVV,ATL,600,556.0,NaN,NaN,cancelado,B
169279,2025-01-05,OO,STL,MSP,1529,1610.0,NaN,NaN,cancelado,B
170061,2025-01-05,OO,RIW,DEN,525,1311.0,NaN,NaN,cancelado,B
170928,2025-01-06,OO,SAN,STS,2055,2139.0,NaN,NaN,cancelado,B
171001,2025-01-06,OO,SEA,BLI,1642,1648.0,NaN,NaN,cancelado,B
174257,2025-01-07,OO,ORD,SLN,1620,1748.0,NaN,NaN,cancelado,B


#### Conclusión: los 111 vuelos son cancelaciones, no errores

- Los 111 vuelos con `DepTime` y sin `DepDelay` son todos cancelados (0,007 % del total).
- En la muestra inspeccionada (10 registros), todos tienen `CancellationCode = B`, son de la aerolínea `OO` y carecen de `TaxiOut`.
- La hipótesis más coherente es que el avión salió de la puerta y la cancelación posterior impidió calcular `DepDelay`. No se ha contrastado con el diccionario del BTS.

**Decisión.** No se imputa `DepDelay`. El retraso de salida se analiza solo en vuelos no cancelados. Los 111 casos se mantienen en el dataset para la tasa de cancelación.

### Verificación del umbral de las columnas de causa de retraso

**Objetivo.** Establecer con los datos si las columnas de causa (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`) se informan con retrasos de llegada superiores a 15 minutos o iguales o superiores a 15 minutos, dado que `ArrDel15` se define con este último criterio.

**Metodología.** Se calcula el valor mínimo de `ArrDelay` entre los vuelos con causa informada y el máximo entre los vuelos `normal` con causa nula. El resultado se contrasta con el diccionario de datos del BTS (`readme.html`).

In [5]:
# 2) Umbral de las columnas de causa de retraso
print("ArrDelay mínimo con causas informadas:", df.loc[df["CarrierDelay"].notna(), "ArrDelay"].min())
print("ArrDelay máximo con causas nulas (vuelos normales):",
      df.loc[df["CarrierDelay"].isna() & (df["estado"] == "normal"), "ArrDelay"].max())

ArrDelay mínimo con causas informadas: 15.0
ArrDelay máximo con causas nulas (vuelos normales): 14.0


 El umbral es **igual o superior a 15 minutos**, no estrictamente superior. Coincide con la definición de `ArrDel15` y corrige la formulación inicial del planteamiento (">15 min"). Los valores nulos en las cinco columnas de causa son, por tanto, información estructural: indican un retraso de llegada inferior a 15 minutos.

No se imputan ceros. El análisis de causas se limita a los 317.266 vuelos con retraso ≥ 15 minutos. Las comparaciones generales usan `ArrDelay`. 

### 4. Valores nulos en `Tail_Number`

Se comprueba si los 4.102 valores nulos de la matrícula se concentran en vuelos cancelados o en alguna aerolínea concreta, para decidir si constituyen un error de registro o un dato estructural.

In [6]:
nulos_tail = df["Tail_Number"].isna()
print("Nulos en Tail_Number:", nulos_tail.sum())
print("\nPor estado:")
print(df.loc[nulos_tail, "estado"].value_counts())
print("\nTop 5 aerolíneas:")
print(df.loc[nulos_tail, "Reporting_Airline"].value_counts().head(5))
print(f"\n% de cancelados sin matrícula: {nulos_tail[df['estado'] == 'cancelado'].mean():.2%}")

Nulos en Tail_Number: 4102

Por estado:
estado
cancelado    4102
Name: count, dtype: int64

Top 5 aerolíneas:
Reporting_Airline
UA    1487
F9     925
OH     858
G4     265
YX     260
Name: count, dtype: int64

% de cancelados sin matrícula: 13.39%


### Conclusión: los nulos de `Tail_Number` son solo de vuelos cancelados

- 4.102 nulos (0,25 % del total), todos en vuelos cancelados (13,39 % de los cancelados).
- UA, F9, OH, G4 y YX concentran el 92,5 %. No se ha normalizado por volumen de cada aerolínea, por lo que no se extraen conclusiones sobre ello.
- Es plausible que la aeronave aún no estuviera asignada al cancelarse el vuelo.

**Decisión.** No se imputa. Los vuelos `normal` tienen la matrícula completa. Como `Tail_Number` no interviene en el análisis, se valorará excluirla del dataset final.

### 5. Formato de las horas (`CRSDepTime`, `DepTime`, `CRSArrTime`, `ArrTime`)

Las horas del BTS se almacenan como número entero con formato HHMM (por ejemplo, 5:25 se codifica como 525). Se comprueba el rango de valores, la presencia del valor 2400 (medianoche), la existencia de minutos no válidos (> 59) y el origen de los nulos, antes de convertirlas a una variable numérica utilizable (minutos desde medianoche).

In [7]:
for col in ["CRSDepTime", "DepTime", "CRSArrTime", "ArrTime"]:
    s = df[col].dropna()
    print(f"{col}: min={s.min():.0f}, max={s.max():.0f}, "
          f"igual a 2400: {(s == 2400).sum()}, "
          f"minutos > 59: {(s % 100 > 59).sum()}, "
          f"nulos: {df[col].isna().sum()}")

print("\nNulos de DepTime por estado:")
print(df.loc[df["DepTime"].isna(), "estado"].value_counts())

CRSDepTime: min=1, max=2359, igual a 2400: 0, minutos > 59: 0, nulos: 0
DepTime: min=1, max=2400, igual a 2400: 135, minutos > 59: 0, nulos: 29560
CRSArrTime: min=1, max=2359, igual a 2400: 0, minutos > 59: 0, nulos: 0
ArrTime: min=1, max=2400, igual a 2400: 836, minutos > 59: 0, nulos: 31252

Nulos de DepTime por estado:
estado
cancelado    29560
Name: count, dtype: int64


#### Conclusión: formato HHMM válido, con 2400 como único caso especial

| Variable | Rango | Valores = 2400 | Minutos > 59 | Nulos |
|---|---|---|---|---|
| `CRSDepTime` | 1 – 2359 | 0 | 0 | 0 |
| `DepTime` | 1 – 2400 | 135 | 0 | 29.560 |
| `CRSArrTime` | 1 – 2359 | 0 | 0 | 0 |
| `ArrTime` | 1 – 2400 | 836 | 0 | 31.252 |

- El valor 2400 solo aparece en horas reales y equivale a las 00:00 del día siguiente.
- Los nulos son estructurales: los de `DepTime` son todos de cancelados, y los de `ArrTime` son 30.640 cancelados y 612 desviados.

**Decisión.** Las horas se convierten a minutos desde medianoche (2400 pasa a 0). El retraso se analiza con `DepDelay` y `ArrDelay` del BTS, sin recalcularlo restando horas. `ArrMin` se deja nulo en los desviados.

### 6. Conversión de las horas a minutos desde medianoche

Se transforman las cuatro variables horarias de formato HHMM a minutos desde medianoche (0–1439), y se validan el rango y el tratamiento del valor 2400.

In [8]:
def hhmm_a_minutos(serie):
    """Convierte HHMM (525 -> 325 min) a minutos desde medianoche; 2400 pasa a 0."""
    return (((serie // 100) * 60 + serie % 100) % 1440).astype("Int16")

df["CRSDepMin"] = hhmm_a_minutos(df["CRSDepTime"])
df["DepMin"] = hhmm_a_minutos(df["DepTime"])
df["CRSArrMin"] = hhmm_a_minutos(df["CRSArrTime"])
df["ArrMin"] = hhmm_a_minutos(df["ArrTime"])

# Llegada de desviados: corresponde a otro aeropuerto, no es comparable con Dest
df.loc[df["estado"] == "desviado", "ArrMin"] = pd.NA

cols_min = ["CRSDepMin", "DepMin", "CRSArrMin", "ArrMin"]

# Validación
print(df[cols_min].agg(["min", "max"]))
assert df[cols_min].max().max() <= 1439, "Hay minutos fuera de rango"
print("\n2400 convertido a 0 en DepMin:", (df.loc[df["DepTime"] == 2400, "DepMin"] == 0).all())
print("2400 convertido a 0 en ArrMin:",
      (df.loc[(df["ArrTime"] == 2400) & (df["estado"] != "desviado"), "ArrMin"] == 0).all())
print("\nNulos:")
print(df[cols_min].isna().sum())

     CRSDepMin  DepMin  CRSArrMin  ArrMin
min          1       0          1       0
max       1439    1439       1439    1439

2400 convertido a 0 en DepMin: True
2400 convertido a 0 en ArrMin: True

Nulos:
CRSDepMin        0
DepMin       29560
CRSArrMin        0
ArrMin       34457
dtype: int64


Las cuatro variables están en el rango 0-1439 y el valor 2400 se convierte correctamente a 0.
Los nulos: `CRSDepMin` 0, `CRSArrMin` 0, `DepMin` 29.560 (cancelados) y `ArrMin` 34.457 (30.640 cancelados + 3.817 desviados).

**Decisión.** Se conservan las columnas originales HHMM hasta el dataset final y se decidirá allí cuáles se eliminan.

## 7. Distribución y valores extremos de `DepDelay` y `ArrDelay`

Se describe la distribución de ambos retrasos en los vuelos `normal`, se cuantifican los adelantos (valores negativos) y los retrasos extremos, y se comprueba la coherencia de `ArrDel15` con `ArrDelay`.

In [9]:
normal = df["estado"] == "normal"

for col in ["DepDelay", "ArrDelay"]:
    s = df.loc[normal, col]
    print(f"--- {col} (vuelos normal) ---")
    print(s.describe(percentiles=[.01, .05, .5, .95, .99, .999]).round(1))
    print(f"Adelantados (< 0): {(s < 0).mean():.1%}")
    print(f"< -60 min: {(s < -60).sum()} | > 300 min: {(s > 300).sum()} | > 1000 min: {(s > 1000).sum()}\n")

# Coherencia entre ArrDel15 y ArrDelay
coherente = (df.loc[normal, "ArrDel15"] == (df.loc[normal, "ArrDelay"] >= 15).astype(float)).all()
print("ArrDel15 coherente con ArrDelay >= 15:", coherente)

# Los 5 retrasos de llegada más grandes
display(df.loc[normal].nlargest(5, "ArrDelay")[["FlightDate", "Reporting_Airline", "Origin", "Dest", "DepDelay", "ArrDelay", "WeatherDelay"]])

--- DepDelay (vuelos normal) ---
count    1611046.0
mean          11.0
std           54.3
min          -56.0
1%           -15.0
5%           -11.0
50%           -3.0
95%           78.0
99%          209.0
99.9%        714.0
max         3403.0
Name: DepDelay, dtype: float64
Adelantados (< 0): 61.8%
< -60 min: 0 | > 300 min: 7500 | > 1000 min: 675

--- ArrDelay (vuelos normal) ---
count    1611046.0
mean           5.0
std           57.1
min          -91.0
1%           -40.0
5%           -30.0
50%           -8.0
95%           80.0
99%          210.0
99.9%        718.0
max         3407.0
Name: ArrDelay, dtype: float64
Adelantados (< 0): 64.4%
< -60 min: 595 | > 300 min: 7544 | > 1000 min: 674

ArrDel15 coherente con ArrDelay >= 15: True


,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay,WeatherDelay
557117,2025-02-24,AA,SJU,PHL,3403.0,3407.0,0.0
57434,2025-01-28,AA,IAH,DFW,3298.0,3282.0,0.0
1104400,2025-03-24,AA,RSW,CLT,3288.0,3275.0,0.0
595533,2025-02-21,AA,MFE,DFW,3092.0,3103.0,0.0
502118,2025-01-03,MQ,MLU,DFW,2983.0,2955.0,0.0


Ambos retrasos tienen una distribución fuertemente asimétrica a la derecha. La mediana es negativa (-3 min en salida y -8 min en llegada) y la mayoría de los vuelos sale (61,8 %) y llega (64,4 %) antes de lo programado, pero la media es mayor que la mediana (11,0 y 5,0 min) por efecto de una cola larga: el percentil 99 se sitúa en torno a 210 min y el máximo llega a 3.407 min (unas 57 horas). Los retrasos superiores a 300 min afectan a unos 7.500 vuelos (0,47 %) y los superiores a 1.000 min a unos 675 (0,04 %); los cinco mayores pertenecen a AA y MQ y no tienen retraso meteorológico asignado (`WeatherDelay` = 0). Del lado opuesto, 595 vuelos llegan más de 60 min antes de lo programado (mínimo de -91 min), sin ningún valor implausible. Además, `ArrDel15` es coherente con `ArrDelay` ≥ 15 en el 100 % de los vuelos normales.

Dado que los valores extremos son registros reales del BTS y no fallos de medida, no se eliminan. Un retraso de varias decenas de horas es compatible con una reprogramación del vuelo, aunque esto es una hipótesis que no se ha contrastado. Para evitar que la cola distorsione el análisis, se usarán la mediana y los percentiles como medidas de resumen, y pruebas no paramétricas (Kruskal-Wallis, Mann-Whitney, Spearman). En los gráficos se limitará el eje al percentil 99 o se usará escala logarítmica. En el dataset final se añadirá un indicador de retraso extremo para poder filtrar estos casos como análisis de sensibilidad.

## 8. Coherencia de los códigos IATA

Se comprueba el formato de los códigos de `Origin` y `Dest`, la ausencia de vuelos con origen igual a destino y la correspondencia con `airports.csv` de OurAirports (presencia y duplicados de `iata_code`), necesaria para obtener las coordenadas del clima.

In [10]:
apt = pd.read_csv("../data/raw/apoyo/airports.csv",
                  usecols=["ident", "type", "name", "latitude_deg", "longitude_deg", "iso_country", "iata_code"])

origen = df["Origin"].astype(str)
destino = df["Dest"].astype(str)
codigos = pd.Index(origen.unique()).union(pd.Index(destino.unique()))

print("Códigos IATA distintos en el BTS:", len(codigos))
print("Formato válido (3 letras mayúsculas):", codigos.str.fullmatch(r"[A-Z]{3}").all())
print("Vuelos con origen == destino:", (origen == destino).sum())

sin_coord = sorted(set(codigos) - set(apt["iata_code"].dropna()))
print("\nCódigos del BTS sin coordenadas en OurAirports:", len(sin_coord), sin_coord)

dup = apt[apt["iata_code"].isin(codigos) & apt["iata_code"].duplicated(keep=False)]
print("\nFilas duplicadas en OurAirports para códigos del BTS:", len(dup))
display(dup.sort_values("iata_code").head(10))

Códigos IATA distintos en el BTS: 333
Formato válido (3 letras mayúsculas): True
Vuelos con origen == destino: 0

Códigos del BTS sin coordenadas en OurAirports: 1 ['PBI']

Filas duplicadas en OurAirports para códigos del BTS: 0


,ident,type,name,latitude_deg,longitude_deg,iso_country,iata_code


Los 333 códigos del BTS tienen formato válido (3 letras mayúsculas), no hay vuelos con origen igual a destino y `iata_code` no está duplicado en OurAirports, por lo que no hace falta filtrar por tipo de aeropuerto. Solo `PBI` (Palm Beach) no tiene coordenadas en OurAirports; se comprobará si está entre los aeropuertos seleccionados para el clima y, si no lo está, no requiere acción.

## 9. Selección de aeropuertos y coordenadas

Se seleccionan los 35 aeropuertos con más vuelos de salida y se obtienen sus coordenadas de OurAirports, necesarias para descargar el clima diario.

In [11]:
top = df["Origin"].value_counts().head(35)
print("Vuelos de los 35 aeropuertos como origen:", f"{top.sum() / len(df):.1%} del total")
print("PBI entre los 35:", "PBI" in top.index)

coords = (apt[apt["iata_code"].isin(top.index)]
          .rename(columns={"iata_code": "airport"})
          [["airport", "name", "latitude_deg", "longitude_deg"]])

print("Aeropuertos con coordenadas:", len(coords), "de 35")
print("Sin coordenadas:", sorted(set(top.index) - set(coords["airport"])))
coords.head()

Vuelos de los 35 aeropuertos como origen: 69.2% del total
PBI entre los 35: False
Aeropuertos con coordenadas: 35 de 35
Sin coordenadas: []


,airport,name,latitude_deg,longitude_deg
38859,ATL,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101
38868,AUS,Austin Bergstrom International Airport,30.197535,-97.662015
38978,BNA,Nashville International Airport,36.124500,-86.678200
38985,BOS,Boston Logan International Airport,42.361970,-71.007900
39024,BWI,Baltimore/Washington International Thurgood Ma...,39.175400,-76.668297


Los 35 aeropuertos con más vuelos de salida concentran el 69,2 % de los vuelos del periodo y todos tienen coordenadas en OurAirports. `PBI`, el único código del BTS sin coordenadas, no está entre ellos, por lo que no requiere ninguna acción.

## 10. Descarga del clima diario (fuente B)

Se descarga con `meteostat` (v1.7.6) el clima diario de enero a marzo de 2025 para los 35 aeropuertos seleccionados, usando sus coordenadas. Se guarda el resultado en `data/raw/clima/` para garantizar la reproducibilidad.

In [12]:
from datetime import datetime
from meteostat import Point, Daily

inicio, fin = datetime(2025, 1, 1), datetime(2025, 3, 31)

partes, fallos = [], []
for fila in coords.itertuples():
    try:
        d = Daily(Point(fila.latitude_deg, fila.longitude_deg), inicio, fin).fetch()
        d = d.reset_index().assign(airport=fila.airport)
        partes.append(d)
    except Exception as e:
        fallos.append((fila.airport, str(e)))

clima = pd.concat(partes, ignore_index=True)

Path("../data/raw/clima").mkdir(parents=True, exist_ok=True)
clima.to_csv("../data/raw/clima/clima_diario_2025q1.csv", index=False)

print("Filas:", len(clima), "| esperadas: 35 x 90 =", 35 * 90)
print("Aeropuertos descargados:", clima["airport"].nunique(), "| fallos:", fallos)
print("Columnas:", list(clima.columns))
print("\n% de nulos por columna:")
print((clima.isna().mean() * 100).round(1))
print("\nFilas por aeropuerto (mínimo y máximo):",
      clima.groupby("airport").size().min(), clima.groupby("airport").size().max())

Filas: 3150 | esperadas: 35 x 90 = 3150
Aeropuertos descargados: 35 | fallos: []
Columnas: ['time', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'airport']

% de nulos por columna:
time         0.0
tavg         0.0
tmin         0.0
tmax         0.0
prcp         0.0
snow        18.7
wdir       100.0
wspd         0.0
wpgt       100.0
pres         0.0
tsun       100.0
airport      0.0
dtype: float64

Filas por aeropuerto (mínimo y máximo): 90 90


Se descargaron las 3.150 filas esperadas (35 aeropuertos × 90 días) sin fallos ni días faltantes. Las variables `wdir`, `wpgt` y `tsun` están vacías al 100 % y se descartan. `tavg`, `tmin`, `tmax`, `prcp`, `wspd` y `pres` están completas. `snow` tiene un 18,7 % de nulos que no se imputan con 0 sin comprobar antes si se concentran en determinados aeropuertos.

### 11. Nulos de `snow`

Se comprueba si los nulos de `snow` se concentran en determinados aeropuertos, lo que indicaría ausencia de medición y no ausencia de nieve.

In [13]:
clima = clima.drop(columns=["wdir", "wpgt", "tsun"])

nulos_snow = clima.groupby("airport")["snow"].apply(lambda s: s.isna().sum())
print("Aeropuertos con snow nulo en los 90 días:", (nulos_snow == 90).sum())
print("Aeropuertos con algún nulo:", (nulos_snow > 0).sum(), "de 35")
print("\nNulos por aeropuerto (solo los que tienen):")
print(nulos_snow[nulos_snow > 0].sort_values(ascending=False))

print("\nDías con nieve > 0:", (clima["snow"] > 0).sum())
print(clima["snow"].describe().round(1))

Aeropuertos con snow nulo en los 90 días: 4
Aeropuertos con algún nulo: 15 de 35

Nulos por aeropuerto (solo los que tienen):
airport
HNL    90
LAX    90
SFO    90
SMF    90
MSY    87
SAN    79
ATL    20
DCA    16
DAL     8
BWI     6
PDX     5
SLC     4
PHX     3
FLL     1
STL     1
Name: snow, dtype: int64

Días con nieve > 0: 282
count    2560.0
mean        0.7
std         2.3
min         0.0
25%         0.0
50%         0.0
75%         0.0
max        23.0
Name: snow, dtype: Float64


Los nulos de `snow` no son aleatorios: en HNL, LAX, SFO y SMF faltan en los 90 días, y en MSY (87) y SAN (79) casi en todos, aeropuertos de clima sin nieve habitual. En otros nueve aeropuertos faltan entre 1 y 20 días. Por ello no se imputan con 0 y `snow` no se usará como base del análisis. La nieve se aproximará con un indicador derivado de precipitación y temperatura (`prcp` > 0 y `tavg` ≤ 0 °C), lo que se indicará como limitación en el informe.

### 12. Unión de vuelos y clima

Se filtran los vuelos con origen en los 35 aeropuertos seleccionados y se unen al clima diario por aeropuerto de origen y fecha (relación muchos a uno). Se valida que no se pierden ni se duplican filas.

In [14]:
df_top = df[df["Origin"].isin(top.index)].copy()

clima_m = clima.rename(columns={"airport": "Origin", "time": "FlightDate"})
clima_m["Origin"] = clima_m["Origin"].astype(df["Origin"].dtype)   # mismo tipo category que en df
clima_m = clima_m.rename(columns={c: f"{c}_orig" for c in ["tavg", "tmin", "tmax", "prcp", "snow", "wspd", "pres"]})

n_antes = len(df_top)
df_m = df_top.merge(clima_m, on=["Origin", "FlightDate"], how="left",
                    validate="m:1", indicator=True)

print("Filas antes:", n_antes, "| después:", len(df_m), "| iguales:", n_antes == len(df_m))
print(df_m["_merge"].value_counts())
print("\nNulos en clima tras el merge:")
print(df_m[[c for c in df_m.columns if c.endswith("_orig")]].isna().mean().mul(100).round(1))
print("\nMáximo de nieve por aeropuerto (top 5):")
print(clima.groupby("airport")["snow"].max().sort_values().tail(5))

Filas antes: 1138858 | después: 1138858 | iguales: True
_merge
both          1138858
left_only           0
right_only          0
Name: count, dtype: int64

Nulos en clima tras el merge:
tavg_orig     0.0
tmin_orig     0.0
tmax_orig     0.0
prcp_orig     0.0
snow_orig    14.8
wspd_orig     0.0
pres_orig     0.0
dtype: float64

Máximo de nieve por aeropuerto (top 5):
airport
DCA    23.0
HNL    <NA>
LAX    <NA>
SFO    <NA>
SMF    <NA>
Name: snow, dtype: Float64


Los 1.138.858 vuelos con origen en los 35 aeropuertos se unieron al clima por aeropuerto y fecha con `validate="m:1"`: el número de filas se mantiene y el 100 % de los vuelos tiene clima. Todas las variables meteorológicas están completas salvo `snow` (14,8 % de nulos), cuyo valor máximo (23, en DCA) es bajo para aeropuertos con nieve invernal frecuente. Se conserva como variable auxiliar, pero la nieve se aproxima con `prcp` y `tavg`.

### 13. Variables derivadas

Se crean la franja horaria de salida programada, el indicador de fin de semana, la categoría de retraso de llegada (con el umbral de 15 min del BTS), un indicador de retraso extremo (> 300 min) y las categorías de precipitación y viento, junto con un indicador de nieve aproximada y otro de clima adverso. Los umbrales son decisiones metodológicas y se documentarán en el informe.

In [15]:
d = df_m.drop(columns="_merge").copy()
d[["Cancelled", "Diverted", "DepDel15", "ArrDel15"]] = d[["Cancelled", "Diverted", "DepDel15", "ArrDel15"]].astype("Int8")

# Tiempo
d["franja_hora"] = pd.cut(d["CRSDepMin"].astype(int), bins=[-1, 359, 719, 1079, 1439],
                          labels=["Madrugada (0-6h)", "Mañana (6-12h)", "Tarde (12-18h)", "Noche (18-24h)"])
d["fin_de_semana"] = d["DayOfWeek"].isin([6, 7])        # BTS: 1 = lunes ... 7 = domingo

# Retraso (vuelos normal; NaN en cancelados y desviados)
d["retraso_cat"] = pd.cut(d["ArrDelay"], bins=[-np.inf, 0, 15, 60, 180, np.inf], right=False,
                          labels=["Adelantado", "0-14 min", "15-59 min", "60-179 min", "≥180 min"])
d["retraso_extremo"] = d["ArrDelay"] > 300              # False también en cancelados/desviados

# Clima en el aeropuerto de origen
d["precip_cat"] = pd.cut(d["prcp_orig"], bins=[-np.inf, 0, 5, 15, np.inf],
                         labels=["Sin lluvia", "Ligera", "Moderada", "Intensa"])
d["viento_cat"] = pd.cut(d["wspd_orig"], bins=[-np.inf, 15, 25, np.inf],
                         labels=["Flojo", "Moderado", "Fuerte"])
d["nieve_aprox"] = (d["prcp_orig"] > 0) & (d["tavg_orig"] <= 0)
d["clima_adverso"] = (d["precip_cat"] == "Intensa") | (d["viento_cat"] == "Fuerte") | d["nieve_aprox"]

# Comprobaciones
print("Forma:", d.shape)
for c in ["franja_hora", "retraso_cat", "precip_cat", "viento_cat"]:
    print(f"\n{c}:\n", d[c].value_counts(normalize=True, dropna=False).mul(100).round(1))
print("\nNieve aproximada:", f"{d['nieve_aprox'].mean():.1%}", "| Clima adverso:", f"{d['clima_adverso'].mean():.1%}")
print("\nArrDelay mediano por clima adverso (vuelos normal):")
print(d[d["estado"] == "normal"].groupby("clima_adverso")["ArrDelay"].median())

Forma: (1138858, 55)

franja_hora:
 franja_hora
Mañana (6-12h)      38.3
Tarde (12-18h)      34.4
Noche (18-24h)      25.5
Madrugada (0-6h)     1.9
Name: proportion, dtype: float64

retraso_cat:
 retraso_cat
Adelantado    62.3
0-14 min      15.9
15-59 min     13.0
60-179 min     5.6
NaN            2.0
≥180 min       1.2
Name: proportion, dtype: float64

precip_cat:
 precip_cat
Sin lluvia    74.5
Ligera        15.8
Moderada       5.8
Intensa        3.9
Name: proportion, dtype: float64

viento_cat:
 viento_cat
Flojo       58.3
Moderado    34.3
Fuerte       7.5
Name: proportion, dtype: float64

Nieve aproximada: 4.1% | Clima adverso: 14.7%

ArrDelay mediano por clima adverso (vuelos normal):
clima_adverso
False   -8.0
True    -1.0
Name: ArrDelay, dtype: float64


Se añadieron franja horaria, fin de semana, categoría de retraso, indicador de retraso extremo, categorías de precipitación y viento, nieve aproximada (4,1 % de los vuelos) y clima adverso (14,7 %). Todas las categorías tienen un tamaño suficiente para el análisis; la menor es la madrugada (1,9 %). El 2,0 % de nulos en `retraso_cat` corresponde a vuelos cancelados y desviados. La diferencia entre las medianas de `ArrDelay` con y sin clima adverso (-1 frente a -8 min) es solo descriptiva y se contrastará en el análisis estadístico.

### 14. Dataset final

Se añaden las coordenadas del aeropuerto de origen (para el mapa del dashboard), se eliminan las columnas redundantes (`Year`, `DayofMonth`, `Tail_Number`, `DestCityName` y las horas en formato HHMM) y se guarda el dataset en `data/processed/`. Se mide su tamaño para decidir el formato de entrega en GitHub.

In [16]:
import os

idx = coords.set_index("airport")
d["lat_orig"] = d["Origin"].astype(str).map(idx["latitude_deg"])
d["lon_orig"] = d["Origin"].astype(str).map(idx["longitude_deg"])

cols_drop = ["Year", "DayofMonth", "Tail_Number", "DestCityName",
             "CRSDepTime", "DepTime", "CRSArrTime", "ArrTime"]
final = d.drop(columns=cols_drop)

Path("../data/processed").mkdir(parents=True, exist_ok=True)
ruta_csv = "../data/processed/vuelos_clima_2025q1.csv"
final.to_csv(ruta_csv, index=False)
print(f"Forma: {final.shape} | CSV: {os.path.getsize(ruta_csv) / 1e6:.0f} MB")

try:
    ruta_pq = "../data/processed/vuelos_clima_2025q1.parquet"
    final.to_parquet(ruta_pq, index=False)
    print(f"Parquet: {os.path.getsize(ruta_pq) / 1e6:.0f} MB")
except ImportError:
    print("Falta pyarrow: ejecuta en la terminal (con el .venv activo) `pip install pyarrow` y vuelve a lanzar la celda.")

print("\nNulos en coordenadas:", final[["lat_orig", "lon_orig"]].isna().sum().sum())

Forma: (1138858, 49) | CSV: 293 MB
Parquet: 25 MB

Nulos en coordenadas: 0


In [17]:
ruta_pq = "../data/processed/vuelos_clima_2025q1.parquet"
final.to_parquet(ruta_pq, index=False)
print(f"Parquet: {os.path.getsize(ruta_pq) / 1e6:.0f} MB")

# Comprobación de ida y vuelta: mismas filas y columnas al releerlo
chk = pd.read_parquet(ruta_pq)
print("Releído:", chk.shape, "| coincide con final:", chk.shape == final.shape)

Parquet: 25 MB
Releído: (1138858, 49) | coincide con final: True


El CSV equivalente pesa 293 MB y supera el límite de 100 MB de GitHub, por lo que se descarta. El formato Parquet ocupa 25 MB, conserva los tipos de datos y se relee sin pérdida de filas ni columnas. Se entrega `vuelos_clima_2025q1.parquet` en `data/processed/`, y el CSV se excluye del repositorio (`.gitignore`). Power BI Desktop lo abre de forma nativa.